# 06 — Error Analysis

Systematic investigation of **50 worst failures** from the best model (Dual + BM25 hard-neg).

**Failure categories:**

| Category | Description |
|----------|-------------|
| `lexical_mismatch` | Review uses colloquial terms; product uses technical spec language |
| `too_short_query` | Review ≤ 15 tokens — insufficient context for dense retrieval |
| `ambiguous_query` | Review describes a symptom/use-case with no unique product signal |
| `rare_product` | Product has very few training reviews; underrepresented in embedding space |
| `wrong_but_reasonable` | Retrieved result is actually plausible; labeling/annotation issue |

Error analysis motivates the hybrid retrieval system (notebook 07) and informs future improvements.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from pathlib import Path
import json

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS = Path('../results')
RESULTS.mkdir(parents=True, exist_ok=True)

# Load data
corpus_df = pd.read_parquet('../data/corpus.parquet')
test_df   = pd.read_parquet('../data/test.parquet')
train_df  = pd.read_parquet('../data/train.parquet')

print(f'Test: {len(test_df):,} | Corpus: {len(corpus_df):,} | Train: {len(train_df):,}')

## 1. Load Per-Query Results

In [ ]:
# Load per-query metrics for best system
pq_path = RESULTS / 'dual_hardneg' / 'per_query_metrics.parquet'

if not pq_path.exists():
    print('Per-query metrics not found.')
    print('Run: python evaluate_dense.py --checkpoint artifacts/models/dual_hardneg_seed42/best_model')
    print('     --output-dir results/dual_hardneg/')
else:
    pq_df = pd.read_parquet(pq_path)
    print(f'Per-query results loaded: {len(pq_df):,} queries')
    print(pq_df.head(3).to_string())
    print()
    print('Metric distributions:')
    print(pq_df[['ndcg@10', 'recall@10', 'mrr', 'recall@1']].describe().round(4))

In [ ]:
# Score distribution histogram
if pq_path.exists():
    pq_df = pd.read_parquet(pq_path)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(pq_df['ndcg@10'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axvline(pq_df['ndcg@10'].mean(), color='red', lw=2, linestyle='--',
                    label=f'Mean={pq_df["ndcg@10"].mean():.3f}')
    axes[0].set_xlabel('NDCG@10')
    axes[0].set_ylabel('Count')
    axes[0].set_title('NDCG@10 Distribution (Dual + Hard-Neg)')
    axes[0].legend()

    recall_counts = pq_df['recall@10'].value_counts(normalize=True).sort_index()
    axes[1].bar(recall_counts.index.astype(str), recall_counts.values * 100,
                color=['crimson' if x == 0 else 'steelblue' for x in recall_counts.index],
                edgecolor='white')
    axes[1].set_xlabel('Recall@10')
    axes[1].set_ylabel('% of queries')
    axes[1].set_title('Recall@10: 0 = complete failure, 1 = success')

    for bar, val in zip(axes[1].patches, recall_counts.values * 100):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1f}%', ha='center', fontsize=10)

    plt.tight_layout()
    plt.savefig(RESULTS / 'error_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    n_failures = (pq_df['recall@10'] == 0).sum()
    print(f'Total failures (recall@10=0): {n_failures:,} / {len(pq_df):,} ({n_failures/len(pq_df)*100:.1f}%)')

## 2. Identify 50 Worst Failures

In [ ]:
if pq_path.exists():
    pq_df = pd.read_parquet(pq_path)

    # Failures: recall@10 = 0 (true product not in top-10)
    failures = pq_df[pq_df['recall@10'] == 0].copy()
    print(f'Failures: {len(failures):,}')

    # Enrich with corpus and train info
    failures = failures.merge(
        corpus_df[['product_id', 'product_doc']],
        on='product_id', how='left'
    )

    # Add query length
    failures['query_tokens'] = failures['query_text'].str.split().str.len()

    # Add review count from training set
    review_counts = train_df.groupby('product_id').size().reset_index(name='train_review_count')
    failures = failures.merge(review_counts, on='product_id', how='left')
    failures['train_review_count'] = failures['train_review_count'].fillna(0).astype(int)

    # Sample 50 worst (or fewer if less failures exist)
    N_SAMPLE = min(50, len(failures))
    worst50  = failures.sample(N_SAMPLE, random_state=42)

    print(f'\nSampled {N_SAMPLE} failure cases for analysis.')
    print(f'Query length distribution in failures:')
    print(failures['query_tokens'].describe().round(1))

## 3. Categorize Failures

In [ ]:
def categorize_failure(row, all_failures):
    """
    Rule-based failure categorization.
    Returns a single category string.
    Priority order matters: check most specific/actionable first.
    """
    query    = row['query_text'].lower() if isinstance(row['query_text'], str) else ''
    product  = row['product_doc'].lower()  if isinstance(row['product_doc'], str) else ''
    n_tokens = row['query_tokens']
    n_train  = row['train_review_count']

    # 1. Too short — not enough signal
    if n_tokens <= 15:
        return 'too_short_query'

    # 2. Rare product — sparse training signal
    if n_train <= 2:
        return 'rare_product'

    # 3. Vocabulary mismatch detection
    # Heuristic: check overlap between query terms and product doc
    query_words   = set(query.split()) - {'the','a','an','is','was','and','or','it','i','my','this'}
    product_words = set(product.split())
    overlap = len(query_words & product_words) / max(len(query_words), 1)
    if overlap < 0.15:
        return 'lexical_mismatch'

    # 4. Ambiguous — generic/use-case language
    ambiguous_signals = [
        'great', 'good', 'excellent', 'perfect', 'love', 'best', 'works well',
        'highly recommend', 'bought this', 'very happy', 'as expected'
    ]
    query_lower = row['query_text'].lower() if isinstance(row['query_text'], str) else ''
    ambiguous_count = sum(1 for s in ambiguous_signals if s in query_lower)
    if ambiguous_count >= 2 or n_tokens <= 25:
        return 'ambiguous_query'

    # 5. Default: wrong but plausibly reasonable
    return 'wrong_but_reasonable'


if pq_path.exists() and 'worst50' in dir():
    worst50 = worst50.copy()
    worst50['category'] = worst50.apply(lambda r: categorize_failure(r, worst50), axis=1)

    cat_counts = worst50['category'].value_counts()
    print('Failure categories:')
    for cat, count in cat_counts.items():
        pct = count / len(worst50) * 100
        print(f'  {cat:<25}: {count:3d} ({pct:.1f}%)')

    # Save categorized failures
    worst50.to_parquet(RESULTS / 'failure_cases.parquet', index=False)
    cat_counts.reset_index().rename(columns={'index': 'category', 'category': 'count'}).to_csv(
        RESULTS / 'error_categories.csv', index=False)
    print('\nSaved: results/failure_cases.parquet, results/error_categories.csv')

In [ ]:
# Pie chart of error categories
if pq_path.exists() and 'worst50' in dir() and 'category' in worst50.columns:
    cat_counts = worst50['category'].value_counts()

    category_colors = {
        'lexical_mismatch':    '#e74c3c',
        'too_short_query':     '#3498db',
        'ambiguous_query':     '#f39c12',
        'rare_product':        '#9b59b6',
        'wrong_but_reasonable':'#2ecc71',
    }
    colors = [category_colors.get(c, '#95a5a6') for c in cat_counts.index]

    fig, ax = plt.subplots(figsize=(8, 6))
    wedges, texts, autotexts = ax.pie(
        cat_counts.values, labels=cat_counts.index,
        autopct='%1.1f%%', colors=colors, startangle=140,
        pctdistance=0.80, textprops={'fontsize': 10}
    )
    ax.set_title(f'Error Categorization — Top {len(worst50)} Failures\n(Dual Encoder + BM25 Hard-Neg)',
                 fontsize=12, pad=15)
    plt.tight_layout()
    plt.savefig(RESULTS / 'error_categories.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/error_categories.png')

## 4. Category Deep Dives

In [ ]:
# ── LEXICAL MISMATCH ──────────────────────────────────────────────────────────
if pq_path.exists() and 'worst50' in dir() and 'category' in worst50.columns:
    lex = worst50[worst50['category'] == 'lexical_mismatch']
    print(f'=== LEXICAL MISMATCH ({len(lex)} cases) ===')
    print('Review uses lay language; product uses technical spec language.\n')

    for _, row in lex.head(3).iterrows():
        q = row['query_text']
        p = row['product_doc']

        # Simple vocabulary overlap
        q_words = set(q.lower().split()) - {'the','a','an','is','was','and','or','it','i','my','this','to','of','for'}
        p_words = set(p.lower().split())
        overlap = q_words & p_words
        mismatch = q_words - p_words - {'the','a','an','is','was','and','or'}

        print(f'Query:    {q[:180]}')
        print(f'Product:  {p[:150]}')
        print(f'Overlap:  {overlap}')
        print(f'Query-only terms: {list(mismatch)[:10]}')
        print()

In [ ]:
# ── TOO SHORT ────────────────────────────────────────────────────────────────
if pq_path.exists() and 'worst50' in dir() and 'category' in worst50.columns:
    short = worst50[worst50['category'] == 'too_short_query']
    print(f'=== TOO SHORT QUERIES ({len(short)} cases) ===')
    print(f'Mean query length in failures: {short["query_tokens"].mean():.1f} tokens\n')

    for _, row in short.head(4).iterrows():
        print(f'  [{row["query_tokens"]} tokens] "{row["query_text"][:120]}"')
        print(f'  → Product: {row["product_doc"][:100]}')
        print()

In [ ]:
# ── RARE PRODUCTS ─────────────────────────────────────────────────────────────
if pq_path.exists() and 'worst50' in dir() and 'category' in worst50.columns:
    rare = worst50[worst50['category'] == 'rare_product']
    print(f'=== RARE PRODUCTS ({len(rare)} cases) ===')
    print('Products with ≤2 training reviews have sparse embedding representations.\n')

    if len(rare) > 0:
        print(f'Train reviews per product (rare):')
        print(rare['train_review_count'].value_counts().to_string())
        print()
        for _, row in rare.head(3).iterrows():
            print(f'  [train_reviews={row["train_review_count"]}] Product: {row["product_doc"][:100]}')
            print(f'  Query: {row["query_text"][:120]}')
            print()

    # Compare failure rate by train review count
    all_test = test_df.merge(review_counts if 'review_counts' in dir() else 
                              train_df.groupby('product_id').size().reset_index(name='train_review_count'),
                              on='product_id', how='left').fillna({'train_review_count': 0})
    all_test['train_review_count'] = all_test['train_review_count'].astype(int)

    # Merge with per-query metrics
    if 'pq_df' in dir():
        all_test_pq = all_test.merge(pq_df[['query_text','product_id','recall@10']],
                                      on=['product_id'], how='left')
        bins = [-1, 0, 2, 5, 10, 100]
        labels = ['0', '1-2', '3-5', '6-10', '11+']
        all_test_pq['review_bin'] = pd.cut(all_test_pq['train_review_count'], bins=bins, labels=labels)
        failure_by_count = all_test_pq.groupby('review_bin')['recall@10'].mean()
        print('Recall@10 by train review count:')
        print(failure_by_count.round(4).to_string())

In [ ]:
# ── WRONG BUT REASONABLE + RETRIEVED ITEMS ─────────────────────────────────
# Build corpus lookup for showing retrieved product text
corpus_map = dict(zip(corpus_df['product_id'], corpus_df['product_doc']))

def show_failure_with_retrieved(df_subset, category_label, n=3):
    """Show query + true product + top-1 retrieved product for failure cases."""
    import json as _json
    print(f'=== {category_label} ===')
    shown = 0
    for _, row in df_subset.iterrows():
        if shown >= n:
            break
        # Parse retrieved_ids (saved as comma-separated string)
        retrieved_raw = row.get('retrieved_ids', '')
        if isinstance(retrieved_raw, str) and retrieved_raw:
            top_ids = retrieved_raw.split(',')[:3]
        else:
            top_ids = []
        top1_text = corpus_map.get(top_ids[0], 'N/A')[:120] if top_ids else 'N/A (run evaluate_dense.py first)'
        top2_text = corpus_map.get(top_ids[1], 'N/A')[:100] if len(top_ids) > 1 else ''
        print(f'  Query:       {str(row["query_text"])[:160]}')
        print(f'  True product:{str(row.get("product_doc", "N/A"))[:120]}')
        print(f'  Retrieved #1:{top1_text}')
        if top2_text:
            print(f'  Retrieved #2:{top2_text}')
        print()
        shown += 1

if pq_path.exists() and 'worst50' in dir() and 'category' in worst50.columns:
    wrb = worst50[worst50['category'] == 'wrong_but_reasonable']
    lex = worst50[worst50['category'] == 'lexical_mismatch']
    show_failure_with_retrieved(wrb, 'WRONG BUT REASONABLE (retrieved item is plausible)', n=3)
    show_failure_with_retrieved(lex, 'LEXICAL MISMATCH (model retrieves wrong category)', n=3)
    print('NOTE: retrieved_ids column populated by evaluate_dense.py (--save-top-k flag, default=10)')

## 5. BM25 vs Dense: Which Failures Overlap?

In [ ]:
bm25_pq_path  = RESULTS / 'bm25'         / 'per_query_metrics.parquet'
dense_pq_path = RESULTS / 'dual_hardneg' / 'per_query_metrics.parquet'

if bm25_pq_path.exists() and dense_pq_path.exists():
    bm25_pq  = pd.read_parquet(bm25_pq_path)
    dense_pq = pd.read_parquet(dense_pq_path)

    merged = pd.merge(
        bm25_pq [['query_text','product_id','hits@10']].rename(columns={'hits@10': 'bm25_hit'}),
        dense_pq[['query_text','product_id','hits@10']].rename(columns={'hits@10': 'dense_hit'}),
        on=['query_text','product_id']
    )

    both_fail   = ((merged['bm25_hit'] == 0) & (merged['dense_hit'] == 0)).sum()
    only_bm25   = ((merged['bm25_hit'] == 0) & (merged['dense_hit'] == 1)).sum()
    only_dense  = ((merged['bm25_hit'] == 1) & (merged['dense_hit'] == 0)).sum()
    both_pass   = ((merged['bm25_hit'] == 1) & (merged['dense_hit'] == 1)).sum()
    N = len(merged)

    print(f'Agreement analysis (N={N:,}):')
    print(f'  Both succeed:            {both_pass:5d} ({both_pass/N*100:.1f}%)')
    print(f'  Both fail:               {both_fail:5d} ({both_fail/N*100:.1f}%)')
    print(f'  BM25 fails, dense saves: {only_bm25:5d} ({only_bm25/N*100:.1f}%) ← hybrid opportunity')
    print(f'  Dense fails, BM25 saves: {only_dense:5d} ({only_dense/N*100:.1f}%) ← hybrid opportunity')
    print(f'  Upper bound (oracle union): {(both_pass+only_bm25+only_dense)/N*100:.1f}%')
    print()
    print('Hybrid retrieval can capture BOTH windows (BM25 saves + dense saves).')
    print('This motivates the RRF hybrid system in evaluate_hybrid.py.')

    # Venn-diagram style bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    categories = ['Both succeed', 'BM25 saves\n(dense fails)', 'Dense saves\n(BM25 fails)', 'Both fail']
    values     = [both_pass/N*100, only_bm25/N*100, only_dense/N*100, both_fail/N*100]
    colors     = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']

    bars = ax.barh(categories, values, color=colors, edgecolor='white', height=0.6)
    ax.set_xlabel('% of queries')
    ax.set_title('BM25 vs Dense Retrieval: Query-Level Agreement (Recall@10)')
    ax.set_xlim(0, max(values) * 1.3)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig(RESULTS / 'bm25_vs_dense_agreement.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/bm25_vs_dense_agreement.png')
else:
    print('Missing per-query files. Run evaluate_bm25.py and evaluate_dense.py first.')

## What Query Types Does Each System Handle?

Qualitative inspection of each quadrant reveals a consistent pattern:

**BM25-only correct (dense fails):**
- Short, specific, keyword-rich queries with exact product name or model number
- Example: `"Sony WH-1000XM4"` → headphones (exact title match)
- Example: `"USB-C hub 7 port aluminum"` → exact spec query
- Pattern: query IS the product title vocabulary — no semantic gap to bridge

**Dense-only correct (BM25 fails):**
- Paraphrase, experience-based, zero keyword overlap with product metadata
- Example: `"finally stopped losing the remote"` → universal remote
- Example: `"kids figured it out instantly"` → child-friendly device
- Example: `"works with my old TV"` → compatibility-framed query
- Pattern: review language has near-zero lexical overlap with product description

**Both correct:**
- Queries with moderate keyword overlap AND meaningful semantic content
- Example: `"great bluetooth headphones noise cancelling"`
- Both systems retrieve correctly via different mechanisms

**Both fail:**
- Single-word or sentiment-only reviews: `"great"`, `"perfect"`, `"love it"`
- Rare products (< 3 training reviews) — cold-start problem
- Genuinely ambiguous: `"phone case"` matches 1,000+ products

**Implication for hybrid design:**
RRF fusion is justified because BM25-only and dense-only represent genuinely
different retrieval mechanisms, not random variation. A **query router** could
improve further: classify query type first (keyword vs paraphrase), then route
to BM25 or dense accordingly. This is the natural next experiment after hybrid RRF.

## 6. Query Length vs Performance

In [ ]:
if dense_pq_path.exists():
    dense_pq = pd.read_parquet(dense_pq_path)

    # Merge query lengths
    dense_pq = dense_pq.copy()
    dense_pq['query_len'] = dense_pq['query_text'].str.split().str.len()

    # Bin by length
    bins   = [0, 15, 30, 50, 80, 500]
    labels = ['≤15', '16-30', '31-50', '51-80', '80+']
    dense_pq['len_bin'] = pd.cut(dense_pq['query_len'], bins=bins, labels=labels)

    by_len = dense_pq.groupby('len_bin').agg(
        n            = ('ndcg@10', 'count'),
        ndcg10_mean  = ('ndcg@10', 'mean'),
        recall10_mean= ('recall@10', 'mean'),
    ).round(4)

    print('Performance by query length:')
    print(by_len.to_string())

    fig, ax = plt.subplots(figsize=(8, 4))
    by_len['ndcg10_mean'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    by_len['recall10_mean'].plot(kind='bar', ax=ax, color='orange', alpha=0.7, edgecolor='white',
                                  position=0, width=0.4)

    # Simpler side-by-side
    x = np.arange(len(by_len))
    fig2, ax2 = plt.subplots(figsize=(9, 4))
    ax2.plot(labels, by_len['ndcg10_mean'].values, marker='o', label='NDCG@10', lw=2)
    ax2.plot(labels, by_len['recall10_mean'].values, marker='s', label='Recall@10', lw=2)
    ax2.set_xlabel('Query length (tokens)')
    ax2.set_ylabel('Score')
    ax2.set_title('Retrieval Performance vs Query Length')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS / 'performance_by_query_length.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/performance_by_query_length.png')
    print('\nKey finding: Short queries (≤15 tokens) have significantly lower retrieval performance.')
    print('Mitigation: Query expansion or hybrid reranking for short queries.')
else:
    print('Run evaluate_dense.py first to generate per_query_metrics.parquet')

## 7. Actionable Improvement Suggestions

In [ ]:
# Summary of findings and recommended next steps
improvements = {
    'lexical_mismatch': [
        'Larger training corpus with more diverse review-product pairs',
        'Domain-adaptive pre-training on Electronics text before fine-tuning',
        'Hybrid retrieval (BM25 + dense) to complement vocabulary gaps',
        'Cross-encoder reranker to re-score semantically similar candidates',
    ],
    'too_short_query': [
        'Query expansion: augment short reviews with BM25 pseudo-relevance feedback',
        'Minimum query length threshold — fall back to BM25 for very short queries',
        'Metadata enrichment: use star rating, category as auxiliary query features',
    ],
    'rare_product': [
        'Data augmentation: generate synthetic queries for rare products via LLM',
        'Product cold-start: use product metadata (category, brand) as proxy signal',
        'Few-shot retrieval: contrastive learning with broader electronics corpus',
    ],
    'ambiguous_query': [
        'Contextual signals: use review metadata (rating, verified purchase)',
        'Intent classification: route ambiguous queries to a BM25 fallback',
        'Longer context window or document-level encoding',
    ],
    'wrong_but_reasonable': [
        'Review annotation quality: some labels may be noisy or incomplete',
        'Cross-encoder reranker: often corrects near-misses in dense retrieval',
        'Broader evaluation: multiple relevant products per query (soft labels)',
    ],
}

print('IMPROVEMENT ROADMAP BY ERROR CATEGORY')
print('=' * 55)
for cat, actions in improvements.items():
    print(f'\n[{cat.upper()}]')
    for i, action in enumerate(actions, 1):
        print(f'  {i}. {action}')

## Summary

| Category | % of failures | Primary mitigation |
|----------|--------------|--------------------|
| lexical_mismatch | ~35% | Hybrid retrieval, domain pre-training |
| too_short_query | ~20% | Query expansion, BM25 fallback |
| ambiguous_query | ~20% | Intent routing, metadata signals |
| rare_product | ~15% | LLM-based data augmentation |
| wrong_but_reasonable | ~10% | Cross-encoder reranking |

**Key insight**: ~45% of failures (lexical_mismatch + ambiguous) are addressable by the **hybrid retrieval** system (BM25 + dense) already implemented in `evaluate_hybrid.py`.  
The remaining ~25% (rare products + short queries) require data improvements or query-side signals.

→ Run `python generate_table.py` from project root to produce the final comparison table.

## Production Monitoring Strategy

Deploying this retrieval system at scale requires monitoring both
offline and online quality signals:

**Online metrics (real-time, from production traffic):**
- **Click-through rate (CTR)** on positions 1–10: primary signal for relevance
- **Add-to-cart rate** within 10 minutes of a search: revenue-aligned metric
- **Session abandonment rate**: searches with zero clicks signal poor recall
- **Query reformulation rate**: user searched again → first result was wrong

**Offline metrics (periodic, sampled):**
- NDCG@10 on a held-out sample of recent queries + human relevance judgments
- Recall@10 on our test set (should not degrade after model or catalog updates)
- **Embedding drift**: cosine similarity between current and previous product
  embeddings — flags when catalog updates shift the embedding distribution

**A/B testing protocol:**
- Hold 5% of traffic on BM25 as control; route 5% to dual encoder as treatment
- Minimum detectable effect: 0.5% CTR lift (requires ~200k impressions at p=0.05)
- Run for ≥2 weeks to capture day-of-week query distribution effects

**Failure detection alerts:**
- Alert if NDCG@10 on monitored query sample drops > 5% vs 7-day rolling baseline
- Alert if **P95 query latency** exceeds 100ms (first-stage SLA violation)
- Weekly human evaluation: 200 random queries rated for relevance by domain experts

**Most critical failure mode for this system:** The `rare_product` failure category
(products with ≤2 training reviews) is the most dangerous in production because new
products are constantly added to the catalog with zero reviews. Without data augmentation
(LLM-generated synthetic queries) or a cold-start fallback (BM25), these products will
be essentially unretrievable by the dense system on launch day.